# Evaluation report and review

After a study: compare runs, explain an Agent order, attribute PnL, check lineage and write a model risk report.

In [ ]:
# A temporary workspace and a synthetic daily recording, so the notebook runs anywhere.
import csv, datetime, json, random
from pathlib import Path
from tempfile import mkdtemp

from sim_platform import Platform

folder = Path(mkdtemp(prefix='market-lab-notebook-'))
rng = random.Random(7)
price = 100.0
recording = folder / 'acme.csv'
with recording.open('w', newline='') as stream:
    out = csv.writer(stream)
    out.writerow(['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
    for day in range(40):
        date = datetime.date(2021, 3, 1) + datetime.timedelta(days=day)
        close = price * (1 + rng.gauss(0.0005, 0.012))
        out.writerow([date.isoformat(), round(price, 2), round(max(price, close) + 0.5, 2),
                      round(min(price, close) - 0.5, 2), round(close, 2), 1_000_000])
        price = close
app = Platform(folder / 'workspace')
profile = app.create_profile('Notebook')
print(recording)

In [ ]:
# A tiny neural network in plain Python: one hidden layer, trained here on the first
# 20 days of returns, then frozen in a model file. It sees only completed bars.
import math

with recording.open() as stream:
    closes = [float(row['Close']) for row in csv.DictReader(stream)]
returns = [b / a - 1 for a, b in zip(closes, closes[1:])]
window = 5
samples = [(returns[i - window:i], 1.0 if returns[i] > 0 else 0.0) for i in range(window, 20)]
weights = [[rng.gauss(0, 0.5) for _ in range(window)] for _ in range(4)]
output = [rng.gauss(0, 0.5) for _ in range(4)]

def forward(x):
    hidden = [math.tanh(sum(w * v * 50 for w, v in zip(row, x))) for row in weights]
    return 1 / (1 + math.exp(-sum(o * h for o, h in zip(output, hidden)))), hidden

for _ in range(300):  # plain gradient descent on the log loss
    for x, y in samples:
        p, hidden = forward(x)
        error = p - y
        for j in range(4):
            grad = error * output[j] * (1 - hidden[j] ** 2)
            output[j] -= 0.05 * error * hidden[j]
            weights[j] = [w - 0.05 * grad * v * 50 for w, v in zip(weights[j], x)]

model_file = folder / 'mlp_model.py'
model_file.write_text(f"""
import math
WEIGHTS = {weights!r}
OUTPUT = {output!r}

def decide(bars, account, context):
    if len(bars) < 6:
        return None
    closes = [bar['close'] for bar in bars[-6:]]
    x = [b / a - 1 for a, b in zip(closes, closes[1:])]
    hidden = [math.tanh(sum(w * v * 50 for w, v in zip(row, x))) for row in WEIGHTS]
    p = 1 / (1 + math.exp(-sum(o * h for o, h in zip(OUTPUT, hidden))))
    return {{'target_position': 100 if p > 0.5 else 0}}
""")
print('model written to', model_file)

In [ ]:
# A configured Agent that reads the market and may trade, answered by RehearsalProvider:
# the whole pipeline runs (tools, orders, journal, lineage) without any model call.
# Swap in a real model with ModelProfile('openai-compatible', ..., base_url=...).
from sim_platform.examples.configured_agent_demo import example_configuration
from sim_platform.models.providers.rehearsal import RehearsalProvider

definition, agent_settings = example_configuration()
definition['requires'] = agent_settings['granted_tools'] = [
    'market.quote', 'orders.submit_market', 'journal.write', 'control.finish', 'model.request']
definition['schedule']['interval_seconds'] = '86400'
app.create_agent(profile, definition)
app.set_model_credential('key', 'unused')
DAY = 86_400 * 10**9
agent_settings.update(symbol='ACME', speed='turn',
                      start_ns=str(25 * DAY + 9 * 3600 * 10**9), end_ns=str(32 * DAY))


def rehearsal():
    return RehearsalProvider('ACME')

In [ ]:
# Every entry on a new account, on the same window, cash and costs: comparable by design.
entries = [
    dict(name='agent', policy='agent', agent_id=definition['agent_id'], revision=1,
         settings=agent_settings, repetitions=1,
         model=dict(provider='openai-responses', model='rehearsal', credential_ref='key')),
    dict(name='rule', policy='rule', quantity=100, hold_ns=5 * DAY),
    dict(name='mlp', policy='model', model=dict(model_file=str(model_file))),
]
with app.open_batch() as batch:
    suite = batch.evaluate_suite([str(recording)], entries, profile=profile, symbol='ACME',
                                 model_provider=rehearsal, baseline='rule', resamples=2000)
evaluation = suite['recordings'][0]['evaluation']
for row in evaluation['entries']:
    stats = row['statistics']['total_return']
    print(f"{row['name']:6} mean return {stats['mean']:+.4f}  runs {stats['n']}  "
          f"deflated Sharpe {row['selection']['deflated_sharpe']}")

In [ ]:
runs = {row['name']: [run['run_id'] for run in row['runs']] for row in evaluation['entries']}
agent_run, rule_run = runs['agent'][0], runs['rule'][0]

timeline = app.decision_timeline(profile, agent_run)
for order in timeline['orders'][:3]:
    print(order['summary'], '<-', [read['summary'] for read in order['reads']])

figures = app.pnl_attribution(profile, rule_run)
market = figures['market']
print('rule: beta', round(market['betas']['market']['beta'], 3),
      'annual alpha', round(market.get('alpha_annual', 0), 4), 'costs', figures['costs']['total'])

In [ ]:
proof = app.verify_run(profile, agent_run)
print('unchanged:', proof['unchanged'], [c['item'] for c in proof['checks']])
diff = app.diff_runs(profile, rule_run, agent_run)
print('differs in', len(diff['settings']), 'settings')
report = app.model_risk_report(profile, [agent_run, rule_run], baseline=rule_run,
                               path=folder / 'review.md')
print((folder / 'review.md').read_text()[:600])